# Stg Layer — comment_reaction (LLM, Incremental, added 2026-11-03)

**Reaction** of the audience to the **trending food review videos** (dashboard v3, page 2 "Video review đang hot"), from `${silver}.stg_comments_latest` → `${silver}.stg_comment_reaction` (1 row / comment).

- Videos: the `TOP_VIDEOS` (30) restaurant review videos (`stg_video_classification.video_type = 'restaurant_review'`, same as `agg_video_trending`) with the most views gained in the last 7 days, computed from `stg_videos` (every read is kept): latest `view_count` − the latest read on or before 7 days earlier; a video with no read before the window counts all its views when it was published in the window, else the views since its first read (an old viral video found by a search is not "trending"). Silver only, so no Gold / serving dependency.
- Comments: the `TOP_COMMENTS_PER_VIDEO` (50) most liked of each of those videos.
- `reaction` (one label): `praise_food` (khen món / quán), `complaint` (chê), `craving` (thèm, muốn đi ăn), `ask_info` (hỏi địa chỉ / giá / giờ), `praise_video` (khen video / reviewer), `other`.
- **Incremental, capped:** only comments not labeled yet (or whose text changed, `text_checksum` includes `PROMPT_VERSION`), most liked first, at most `MAX_COMMENTS` per run (100; the refresh job runs every 3 h, so ≤ ~600 `ai_query` calls a day, datle 2026-11-03).

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("max_comments", "100")
MAX_COMMENTS = int(dbutils.widgets.get("max_comments"))

VIDEOS = f"{SILVER}.stg_videos"                    # every read of every video (snapshot_ts)
COMMENTS = f"{SILVER}.stg_comments_latest"         # latest read per comment
CLASSES = f"{SILVER}.stg_video_classification"
TARGET = f"{SILVER}.stg_comment_reaction"
TOP_VIDEOS = 30
TOP_COMMENTS_PER_VIDEO = 50
TREND_DAYS = 7
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # same model as comment_sentiment
MAX_TEXT_CHARS = 600
REACTIONS = ["praise_food", "complaint", "craving", "ask_info", "praise_video", "other"]

PROMPT = (
    "You read one Vietnamese YouTube comment under a food review video. Pick the ONE reaction that fits best: "
    "praise_food = praises the food, drinks, price or the place; "
    "complaint = complains about taste, price, hygiene, service, or says it is an ad / overrated; "
    "craving = wants to eat it, wants to go, looks delicious, tags a friend to go together; "
    "ask_info = asks for the address, price, opening hours or the name of the place or dish; "
    "praise_video = praises or talks to the YouTuber, the video, editing or voice, not the food; "
    "other = anything else."
)
RESPONSE_FORMAT = ('{"type":"json_schema","json_schema":{"name":"comment_reaction","schema":{"type":"object",'
                   '"properties":{"reaction":{"type":"string","enum":' + str(REACTIONS).replace("'", '"') + '}},'
                   '"required":["reaction"]},"strict":true}}')
RESULT_SCHEMA = "STRUCT<reaction: STRING>"
import hashlib
PROMPT_VERSION = hashlib.md5((MODEL + PROMPT + RESPONSE_FORMAT).encode()).hexdigest()[:8]
print(f"Source: {COMMENTS} · target: {TARGET} · top {TOP_VIDEOS} videos x {TOP_COMMENTS_PER_VIDEO} comments · "
      f"max {MAX_COMMENTS} comments / run · model {MODEL}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET} (
        comment_id     STRING,
        video_id       STRING,
        reaction       STRING,
        text_checksum  STRING,
        raw_response   STRING,
        model_name     STRING,
        analyzed_at    TIMESTAMP
    )
    USING DELTA
    COMMENT 'LLM reaction (praise_food / complaint / craving / ask_info / praise_video / other) of the most liked comments on the trending food review videos (1 row per comment).'
""")

# --- Trending food review videos: views gained over the last TREND_DAYS days, from every read in stg_videos ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW trending_videos AS
    WITH reads AS (
        SELECT video_id, snapshot_date, snapshot_ts, view_count, published_at FROM {VIDEOS} WHERE view_count IS NOT NULL
    ),
    w AS (SELECT MAX(snapshot_date) AS d, date_sub(MAX(snapshot_date), {TREND_DAYS}) AS start FROM reads),
    latest AS (SELECT video_id, max_by(view_count, snapshot_ts) AS views_now, min_by(view_count, snapshot_ts) AS first_views,
                      max(published_at) AS published_at
               FROM reads GROUP BY video_id),
    base AS (   -- last read on or before the window start; none = the video appeared in the window
        SELECT r.video_id, max_by(r.view_count, r.snapshot_ts) AS views_then
        FROM reads r CROSS JOIN w WHERE r.snapshot_date <= w.start GROUP BY r.video_id
    )
    -- no read before the window: published in the window -> all its views; found later by a search -> views since the first read
    SELECT l.video_id,
           l.views_now - CASE WHEN b.views_then IS NOT NULL THEN b.views_then
                              WHEN to_date(l.published_at) > w.start THEN 0 ELSE l.first_views END AS views_7d
    FROM latest l
    CROSS JOIN w
    JOIN {CLASSES} c ON c.video_id = l.video_id AND c.video_type = 'restaurant_review'
    LEFT JOIN base b ON b.video_id = l.video_id
    ORDER BY views_7d DESC
    LIMIT {TOP_VIDEOS}
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW comments_todo AS
    WITH top_comments AS (
        SELECT c.comment_id, c.video_id, c.text, c.like_count,
               md5(concat(coalesce(c.text, ''), '{PROMPT_VERSION}')) AS text_checksum,
               ROW_NUMBER() OVER (PARTITION BY c.video_id ORDER BY c.like_count DESC NULLS LAST, c.comment_id) AS rn
        FROM {COMMENTS} c JOIN trending_videos v ON v.video_id = c.video_id
        WHERE length(trim(coalesce(c.text, ''))) > 1
    )
    SELECT s.comment_id, s.video_id, s.text, s.like_count, s.text_checksum FROM top_comments s
    LEFT JOIN {TARGET} t ON t.comment_id = s.comment_id
    WHERE s.rn <= {TOP_COMMENTS_PER_VIDEO} AND (t.comment_id IS NULL OR t.text_checksum <> s.text_checksum)
    ORDER BY s.like_count DESC NULLS LAST, s.comment_id
    LIMIT {MAX_COMMENTS}
""")
todo = spark.table("comments_todo").count()
display(spark.table("trending_videos").limit(10))
print(f"Comments to label this run: {todo}")

In [ ]:
if todo > 0:
    prompt_sql = PROMPT.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {TARGET} AS t
        USING (
            SELECT comment_id, video_id, r.reaction, text_checksum, raw_response,
                   '{MODEL}' AS model_name, current_timestamp() AS analyzed_at
            FROM (
                SELECT *, from_json(raw_response, '{RESULT_SCHEMA}') AS r
                FROM (
                    SELECT comment_id, video_id, text_checksum,
                           ai_query('{MODEL}', concat('{prompt_sql}', '\\n\\nCOMMENT: ', left(text, {MAX_TEXT_CHARS})),
                                    responseFormat => '{RESPONSE_FORMAT}') AS raw_response
                    FROM comments_todo
                )
            )
        ) AS s
        ON t.comment_id = s.comment_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Labeled {todo} comment(s)")
else:
    print("Nothing new to label ✅")

s = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT comment_id) AS grain, COUNT_IF(reaction IS NULL) AS unparsed FROM {TARGET}").collect()[0]
display(spark.sql(f"SELECT reaction, COUNT(*) AS n FROM {TARGET} GROUP BY 1 ORDER BY 2 DESC"))
print(f"📊 {TARGET}: {s['n']} comments · unparsed {s['unparsed']} " + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
assert s["n"] == s["grain"], "duplicate comment_id"